# 02 — Feature Engineering
**Goal:** Build predictive features from raw inventory data for at-risk SKU classification.
**Input:** `data/processed/01_cleaned_with_risk.csv`
**Output:** `data/processed/02_features.csv` with 15 engineered features

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/01_cleaned_with_risk.csv')
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (10000, 22)
Columns: ['Product ID', 'Product Name', 'Product Category', 'Product Description', 'Price', 'Stock Quantity', 'Warranty Period', 'Product Dimensions', 'Manufacturing Date', 'Expiration Date', 'SKU', 'Product Tags', 'Color/Size Variations', 'Product Ratings', 'Manufacturing_Date', 'Expiration_Date', 'days_to_expiry', 'Shelf_Life', 'Risk_Level', 'inventory_value', 'stock_bucket', 'expiry_bucket']


In [2]:
# Parse dates and derive Shelf_Life + days_to_expiry from raw date columns
df['Manufacturing_Date'] = pd.to_datetime(df['Manufacturing Date'], errors='coerce')
df['Expiration_Date']    = pd.to_datetime(df['Expiration Date'],    errors='coerce', dayfirst=True)

# Fixed reference date for reproducibility
today = pd.Timestamp('2024-01-01')

df['days_to_expiry'] = (df['Expiration_Date'] - today).dt.days
df['Shelf_Life']     = (df['Expiration_Date'] - df['Manufacturing_Date']).dt.days.clip(lower=1)

# Sanity check
print("Date parsing check:")
print(df[['Manufacturing_Date', 'Expiration_Date', 'days_to_expiry', 'Shelf_Life']].head(10))
print(f"\ndays_to_expiry — min: {df['days_to_expiry'].min()}, max: {df['days_to_expiry'].max()}")
print(f"Shelf_Life     — min: {df['Shelf_Life'].min()},  max: {df['Shelf_Life'].max()}")
print(f"Already expired (<=0): {(df['days_to_expiry'] <= 0).sum():,} SKUs")

Date parsing check:
  Manufacturing_Date Expiration_Date  days_to_expiry  Shelf_Life
0         2023-01-01      2026-01-01             731        1096
1         2023-03-15      2025-01-01             366         658
2         2023-03-15      2026-01-01             731        1023
3         2023-01-01      2026-01-01             731        1096
4         2023-07-30      2026-01-01             731         886
5         2023-01-01      2024-01-01               0         365
6         2023-01-01      2026-01-01             731        1096
7         2023-01-01      2026-01-01             731        1096
8         2023-01-01      2025-01-01             366         731
9         2023-01-01      2026-01-01             731        1096

days_to_expiry — min: 0, max: 731
Shelf_Life     — min: 155,  max: 1096
Already expired (<=0): 3,320 SKUs


In [3]:
# FEATURE 1: Stock pressure — how close to max (100)?
df['stock_pressure'] = df['Stock Quantity'] / 100

# FEATURE 2: Expiry urgency — higher = more urgent
# Handles negatives (already expired) cleanly via clip
df['expiry_urgency'] = 1 - (df['days_to_expiry'].clip(-731, 731) / 731)
df['expiry_urgency'] = df['expiry_urgency'].clip(0, 1)

# FEATURE 3: Combined risk score
df['risk_score'] = df['stock_pressure'] * df['expiry_urgency']

# FEATURE 4: Is expired flag
df['is_expired'] = (df['days_to_expiry'] <= 0).astype(int)

# FEATURE 5: High stock flag
df['high_stock_flag'] = (df['Stock Quantity'] > 75).astype(int)

# FEATURE 6: Near expiry flag (includes all expired since days_to_expiry <= 0 < 180)
df['near_expiry_flag'] = (df['days_to_expiry'] < 180).astype(int)

print("Features 1–6 done.")
print(df[['stock_pressure','expiry_urgency','risk_score',
          'is_expired','high_stock_flag','near_expiry_flag']].describe().round(3))

Features 1–6 done.
       stock_pressure  expiry_urgency  risk_score  is_expired  \
count       10000.000       10000.000   10000.000   10000.000   
mean            0.506           0.498       0.252       0.332   
std             0.289           0.409       0.278       0.471   
min             0.010           0.000       0.000       0.000   
25%             0.250           0.000       0.000       0.000   
50%             0.510           0.499       0.170       0.000   
75%             0.760           1.000       0.420       1.000   
max             1.000           1.000       1.000       1.000   

       high_stock_flag  near_expiry_flag  
count        10000.000         10000.000  
mean             0.253             0.332  
std              0.435             0.471  
min              0.000             0.000  
25%              0.000             0.000  
50%              0.000             0.000  
75%              1.000             1.000  
max              1.000             1.000  


In [4]:
# FEATURE 7: Price tier (low / mid / high tertiles)
df['price_tier'] = pd.qcut(df['Price'], q=3, labels=['low', 'mid', 'high'])
df['price_tier_enc'] = df['price_tier'].map({'low': 0, 'mid': 1, 'high': 2})

# FEATURE 8: Categorical encodings
df['category_enc'] = df['Product Category'].astype('category').cat.codes
df['warranty_enc']  = df['Warranty Period'].astype('category').cat.codes

# FEATURE 9: Financial exposure (stock qty × price)
df['financial_exposure'] = df['Stock Quantity'] * df['Price']

# FEATURE 10: Overhang ratio — units beyond safe threshold (50) relative to total
df['overhang_ratio'] = (
    (df['Stock Quantity'] - 50).clip(0) / df['Stock Quantity'].clip(1)
)

print("Features 7–10 done.")
print(df[['price_tier_enc','category_enc','warranty_enc',
          'financial_exposure','overhang_ratio']].describe().round(3))

Features 7–10 done.
       category_enc  warranty_enc  financial_exposure  overhang_ratio
count     10000.000     10000.000           10000.000       10000.000
mean          0.996         1.014           12883.242           0.157
std           0.815         0.818           11094.105           0.185
min           0.000         0.000              17.860           0.000
25%           0.000         0.000            3761.435           0.000
50%           1.000         1.000            9639.360           0.020
75%           2.000         2.000           19665.802           0.342
max           2.000         2.000           49605.000           0.500


In [5]:
# FEATURE 11: % shelf life remaining (0 if expired, 1 if brand new)
df['pct_shelf_life_remaining'] = (
    df['days_to_expiry'] / df['Shelf_Life'].clip(lower=1)
).clip(0, 1)

# FEATURE 12: Shelf life in days (clean alias)
df['shelf_life_days'] = df['Shelf_Life']

# FEATURE 13: Price per remaining day — financial urgency signal
# Uses days_to_expiry clipped at 1 to avoid divide-by-zero
df['price_per_day_shelf'] = df['Price'] / df['days_to_expiry'].clip(lower=1)

# FEATURE 14: Inventory value (total financial exposure per SKU)
df['inventory_value'] = df['Stock Quantity'] * df['Price']

# FEATURE 15: Volume (physical footprint — placeholder if dimensions not in dataset)
if all(c in df.columns for c in ['Length', 'Width', 'Height']):
    df['volume_cm3'] = df['Length'] * df['Width'] * df['Height']
else:
    df['volume_cm3'] = 0
    print("Note: Length/Width/Height not found — volume_cm3 set to 0")

print("Features 11–15 done.")
print(df[['pct_shelf_life_remaining','shelf_life_days',
          'price_per_day_shelf','inventory_value','volume_cm3']].describe().round(3))

Note: Length/Width/Height not found — volume_cm3 set to 0
Features 11–15 done.
       pct_shelf_life_remaining  shelf_life_days  price_per_day_shelf  \
count                 10000.000        10000.000            10000.000   
mean                      0.441          639.731               84.630   
std                       0.323          311.270              144.545   
min                       0.000          155.000                0.014   
25%                       0.000          365.000                0.350   
50%                       0.556          658.000                0.684   
75%                       0.702          886.000              130.912   
max                       0.825         1096.000              499.970   

       inventory_value  volume_cm3  
count        10000.000     10000.0  
mean         12883.242         0.0  
std          11094.105         0.0  
min             17.860         0.0  
25%           3761.435         0.0  
50%           9639.360         0.0  
75% 

In [6]:
# Target: based on financial exposure + shelf life ONLY
# High financial exposure AND short shelf life remaining = at risk
exposure_threshold = df['financial_exposure'].quantile(0.55)
shelf_threshold = df['pct_shelf_life_remaining'].quantile(0.45)

df['is_at_risk'] = (
    (df['financial_exposure'] >= exposure_threshold) &
    (df['pct_shelf_life_remaining'] <= shelf_threshold)
).astype(int)

print(f"Exposure threshold: ${exposure_threshold:,.0f}")
print(f"Shelf life threshold: {shelf_threshold:.3f}")
print(df['is_at_risk'].value_counts())
print(df['is_at_risk'].value_counts(normalize=True).round(3))

Exposure threshold: $11,176
Shelf life threshold: 0.556
is_at_risk
0    7510
1    2490
Name: count, dtype: int64
is_at_risk
0    0.751
1    0.249
Name: proportion, dtype: float64


In [7]:
ALL_FEATURES = [
    'stock_pressure', 'expiry_urgency', 'risk_score',
    'is_expired', 'high_stock_flag', 'near_expiry_flag',
    'price_tier_enc', 'category_enc', 'warranty_enc',
    'financial_exposure', 'overhang_ratio',
    'pct_shelf_life_remaining', 'shelf_life_days',
    'price_per_day_shelf', 'inventory_value', 'volume_cm3'
]

print("=== ALL 15 FEATURES SUMMARY ===")
print(df[ALL_FEATURES].describe().round(3))
print(f"\nFinal shape: {df.shape}")
print(f"At-risk SKUs: {df['is_at_risk'].sum():,} ({df['is_at_risk'].mean():.1%})")

df.to_csv('../data/processed/02_features.csv', index=False)
print("\nSaved → data/processed/02_features.csv")
print("Saved. At-risk count:", df['is_at_risk'].sum())

=== ALL 15 FEATURES SUMMARY ===
       stock_pressure  expiry_urgency  risk_score  is_expired  \
count       10000.000       10000.000   10000.000   10000.000   
mean            0.506           0.498       0.252       0.332   
std             0.289           0.409       0.278       0.471   
min             0.010           0.000       0.000       0.000   
25%             0.250           0.000       0.000       0.000   
50%             0.510           0.499       0.170       0.000   
75%             0.760           1.000       0.420       1.000   
max             1.000           1.000       1.000       1.000   

       high_stock_flag  near_expiry_flag  category_enc  warranty_enc  \
count        10000.000         10000.000     10000.000     10000.000   
mean             0.253             0.332         0.996         1.014   
std              0.435             0.471         0.815         0.818   
min              0.000             0.000         0.000         0.000   
25%              0.000

### Feature Engineering Notes

- **days_to_expiry** derived from `Expiration Date - 2024-01-01` (fixed ref date for reproducibility)
- **Shelf_Life** derived from `Expiration Date - Manufacturing Date`
- Most SKUs may show `days_to_expiry <= 0` (already expired at ref date) → `expiry_urgency` will be 1.0 for all → this is real signal, not a bug
- `price_per_day_shelf` will be very high for expired items — winsorize in notebook 03 if needed
- `volume_cm3` = 0 placeholder since dataset has no dimension columnssm